In [1]:
import os
import json
import torch
import numpy as np
import torchvision
import time  
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import torchvision.transforms.functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torch.optim as optim
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. SETUP & EXACT PATHS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Initializing on device: {device}")

TRAIN_JSON = "/kaggle/input/datasets/hades1998/subsampling-json/subsampled_train_data.json"
TRAIN_DIR = "/kaggle/input/datasets/hrushi1998/vr-mini-proj-dataset/Trimmed_Dataset/trimmed_train_data"
VAL_DIR = "/kaggle/input/datasets/hrushi1998/vr-mini-proj-dataset/Trimmed_Dataset/trimmed_val_data"
VAL_JSON = os.path.join(VAL_DIR, "processed_val_data.json")

BEST_MODEL_PATH = "/kaggle/working/maskrcnn_finetune.pth"

# ==========================================
# 2. DATASET CLASS 
# ==========================================
class MasterJSONDataset(Dataset):
    def __init__(self, json_file, img_dir):
        self.img_dir = img_dir
        with open(json_file, 'r') as f:
            self.full_data = json.load(f)["data"]
        
        self.top_5_categories = [1, 8, 7, 2, 9] 
        self.label_map = {cat_id: idx + 1 for idx, cat_id in enumerate(self.top_5_categories)}

    def __len__(self): return len(self.full_data)

    def __getitem__(self, idx):
        item = self.full_data[idx]
        img_path = os.path.join(self.img_dir, item["image_path"])
        if not os.path.exists(img_path): return None
            
        img = Image.open(img_path).convert("RGB")
        width, height = img.size
        boxes, labels, masks = [], [], []
        
        categories = item.get("item_categories", [])
        bboxes = item.get("detection_bboxes", [])
        polygons = item.get("segmentation_polygons", [])
        
        for i in range(len(categories)):
            cat_id = categories[i]
            if cat_id in self.label_map:
                box = bboxes[i]
                if box[2] <= box[0] or box[3] <= box[1]: continue 
                
                labels.append(self.label_map[cat_id])
                boxes.append(box)
                
                mask = Image.new('L', (width, height), 0)
                draw = ImageDraw.Draw(mask)
                for poly in polygons[i]:
                    draw.polygon(poly, outline=1, fill=1)
                masks.append(np.array(mask))
                    
        if len(boxes) == 0: return None
            
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8),
            "image_id": torch.tensor([idx]),
            "area": torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes]),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64)
        }
        return F.to_tensor(img), target

def custom_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return ()
    return tuple(zip(*batch))

train_loader = DataLoader(MasterJSONDataset(TRAIN_JSON, TRAIN_DIR), batch_size=4, shuffle=True, num_workers=2, pin_memory=True, collate_fn=custom_collate)
val_loader = DataLoader(MasterJSONDataset(VAL_JSON, VAL_DIR), batch_size=4, shuffle=False, num_workers=2, pin_memory=True, collate_fn=custom_collate)

# ==========================================
# 3. FULL FINE-TUNING ARCHITECTURE
# ==========================================
print("\n🧠 Initializing Mask R-CNN for FULL FINE-TUNING (Unfrozen Backbone)...")

model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT', min_size=640, max_size=640)

NUM_CLASSES = 6 
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, NUM_CLASSES)

model.to(device)

# ==========================================
# 4. ADVANCED TRAINING LOOP
# ==========================================
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=0.00001, weight_decay=0.0005)

# 🚀 Dynamic Scheduler: Automatically adjusts to your total epochs!
num_epochs = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

scaler = torch.cuda.amp.GradScaler()
best_val_loss = float('inf')

print(f"\n Starting {num_epochs}-Epoch Fine-Tuning...")

for epoch in range(num_epochs):
    epoch_start_time = time.time()  # ⏱️ Start epoch timer

    model.train() 
    # tqdm automatically handles the live ETA countdown on the right side of the bar!
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]")
    train_loss = 0
    
    for batch in train_bar:
        if not batch: continue
        images, targets = batch
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        scaler.scale(losses).backward()
        
        # Gradient Clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += losses.item()
        
        current_lr = optimizer.param_groups[0]['lr']
        train_bar.set_postfix({'Loss': f"{losses.item():.4f}", 'LR': f"{current_lr:.1e}"})
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Smoothly adjust learning rate for the next epoch
    scheduler.step()
    
    # Validation Phase
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            if not batch: continue
            images, targets = batch
            images = [img.to(device, non_blocking=True) for img in images]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
            
            with torch.cuda.amp.autocast():
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    epoch_end_time = time.time()  # ⏱️ Stop epoch timer
    epoch_duration = epoch_end_time - epoch_start_time
    minutes = int(epoch_duration // 60)
    seconds = int(epoch_duration % 60)
    
    # Permanent log with exact time taken
    print(f" Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.1e} | Time Taken: {minutes}m {seconds}s")
    
    if avg_val_loss < best_val_loss:
        print(f"New Best Validation Loss! Saving model to {BEST_MODEL_PATH}")
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss
        }, BEST_MODEL_PATH)

print("\n Fine-Tuning Complete! Model safely saved.")

🚀 Initializing on device: cuda

🧠 Initializing Mask R-CNN for FULL FINE-TUNING (Unfrozen Backbone)...
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 181MB/s]



 Starting 10-Epoch Fine-Tuning...


Epoch 1/10 [TRAIN]: 100%|██████████| 9011/9011 [52:44<00:00,  2.85it/s, Loss=0.3697, LR=1.0e-05]


 Epoch 1 | Train Loss: 0.3568 | Val Loss: 0.3045 | LR: 1.0e-05 | Time Taken: 68m 37s
New Best Validation Loss! Saving model to /kaggle/working/maskrcnn_finetune.pth


...

 Epoch 8 | Train Loss: 0.1484 | Val Loss: 0.3047 | LR: 2.9e-06 | Time Taken: 67m 57s


Epoch 9/10 [TRAIN]: 100%|██████████| 9011/9011 [51:42<00:00,  2.90it/s, Loss=0.2099, LR=1.9e-06]


 Epoch 9 | Train Loss: 0.1407 | Val Loss: 0.3111 | LR: 1.9e-06 | Time Taken: 67m 17s


Epoch 10/10 [TRAIN]: 100%|██████████| 9011/9011 [51:41<00:00,  2.91it/s, Loss=0.1727, LR=1.2e-06]


 Epoch 10 | Train Loss: 0.1356 | Val Loss: 0.3211 | LR: 1.2e-06 | Time Taken: 67m 17s

 Fine-Tuning Complete! Model safely saved.
